# Data Quality Monitoring System for E-Commerce Operations

## Project Overview

This project presents the development of a Data Quality Monitoring System using the **Olist Brazilian E-Commerce Public Dataset**. The objective is to simulate a real-world business analytics workflow by identifying, assessing, and improving the quality of transactional data before it is used for reporting and decision-making.

The project follows an end-to-end data analytics pipeline, beginning with data preparation in **Python**, followed by data modelling and quality analysis in **PostgreSQL**, and concluding with interactive **Power BI** dashboards that monitor key data quality metrics and business performance indicators.

Throughout the project, data quality dimensions such as **completeness, accuracy, consistency, validity, uniqueness, and timeliness** are evaluated. The datasets are cleaned, standardized, validated, and transformed into analysis-ready data to support reliable business intelligence and operational reporting.

This project demonstrates practical skills in data cleaning, exploratory data analysis, SQL development, data quality assessment, and dashboard development while following industry-standard data analytics practices.

This script focuses on the preparation of the geolocation dataset as part of the Data Quality Monitoring System for e-commerce operations. It performs data inspection, profiling, cleaning, and validation to identify and address data quality issues before the dataset is used for downstream analysis.

### Import The Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re 
import psycopg2
from sqlalchemy import create_engine 
from pathlib import Path

### Load the dataset

In [2]:
geolocation_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_datasets\raw\olist_geolocation_dataset.csv")

## **GEOLOCATION DATASET**

### 1. Inspect the dataset

In [4]:
# The first 5 rows of the geolocation dataset.
geolocation_df.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [5]:
# The number of rows and columns in the geolocation dataset.
geolocation_df.shape

(1000163, 5)

In [6]:
# The data types of each column in the geolocation dataset.
geolocation_df.dtypes

geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

The dataset contains appropriate data types for all five attributes. The ZIP code prefix is stored as an integer, latitude and longitude are stored as floating-point values to preserve geographic precision, and the city and state fields are stored as text. However, the geolocation_zip_code_prefix is currently stored as an integer, despite functioning as an identifier rather than a numerical value. To ensure consistency with related datasets and preserve leading zeros, this column should be converted to a string during the data cleaning stage.

In [7]:
# Summary statistics for numerical columns
geolocation_df.describe()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng
count,1.000163e+06,1.000163e+06,1.000163e+06
mean,3.657417e+04,-2.117615e+01,-4.639054e+01
std,3.054934e+04,5.715866e+00,4.269748e+00
min,1.001000e+03,-3.660537e+01,-1.014668e+02
25%,1.107500e+04,-2.360355e+01,-4.857317e+01
50%,2.653000e+04,-2.291938e+01,-4.663788e+01
75%,6.350400e+04,-1.997962e+01,-4.376771e+01
max,9.999000e+04,4.506593e+01,1.211054e+02


- **Count:** All three numerical columns (geolocation_zip_code_prefix, geolocation_lat, and geolocation_lng) have 1,000,163 records. Since the counts are identical, this suggests there are no missing values in these numerical fields.
- **ZIP code prefix:** The values range from **1,001 to 99,990**, which appears reasonable for Brazilian ZIP code prefixes. Nothing looks suspicious from the summary statistics alone.
- **Latitude:** This is where an anomaly appears. Brazil is located approximately between **34°S and 5°N**. Your dataset contains a **maximum latitude of 45.07°**, which is far north of Brazil and therefore suspicious.
- **Longitude:** Brazil's longitude is approximately between **74°W and 34°W**. The dataset contains a **minimum longitude of -101.47° and a maximum longitude of 121.11°**, both of which are well outside Brazil's geographic boundaries. These are likely erroneous coordinate values or outliers.
- **Mean, median and quartiles:** These look sensible. The mean and median are fairly close for both latitude and longitude, suggesting there isn't a severe skew in the majority of the data despite the presence of extreme values.

In [9]:
# Summary statistics for categorical columns
geolocation_df.describe(include='object')

,geolocation_city,geolocation_state
count,1000163,1000163
unique,8011,27
top,sao paulo,SP
freq,135800,404268


**Count:** Both geolocation_city and geolocation_state have 1,000,163 records, matching the total number of rows. This suggests there are no missing values in either categorical column.

**Unique values:**
- geolocation_city has 8,011 unique city names. This is plausible, although it doesn't necessarily mean there are 8,011 distinct Brazilian cities. The high count may be due to inconsistent naming conventions (e.g., differences in capitalization, accents, abbreviations, or spelling), which is something to investigate later.
- geolocation_state has 27 unique values, which is exactly what we'd expect for Brazil's 26 states plus the Federal District (DF). This is a good sign.

**Most frequent values:**
- The most common city is "sao paulo" with 135,800 occurrences.
- The most common state is "SP" with 404,268 occurrences.
These results are expected because São Paulo is Brazil's largest city and state, so there is nothing unusual here.

In [10]:
# Check for missing values
geolocation_df.isnull().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [12]:
# Check for duplicate values
for column in geolocation_df.columns:
    duplicates = geolocation_df[column].duplicated().sum()
    print(f"{column}: {duplicates}")

geolocation_zip_code_prefix: 981148
geolocation_lat: 282803
geolocation_lng: 282550
geolocation_city: 992152
geolocation_state: 1000136
